# Day 4 - Code breaks, block 1: convolutions

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the number marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough**. You cannot break anything - if something looks strange: *Runtime → Restart session*,
then run the first two cells again.

**Data:** small hand-made tensors and one *MNIST* image - a handwritten digit (28 × 28 pixels, grey).

## Setup - run these two cells first
The first cell loads PyTorch and the MNIST digits (about 10 MB, downloaded once).

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets

digits = datasets.MNIST("data", train=True, download=True)
print("ready - PyTorch", torch.__version__, "-", len(digits), "MNIST digits")

The second cell defines small helpers. **Just run it - you do not need to read it.**
- `get_image(nr)` - one MNIST digit as a tensor of shape (1, 1, 28, 28), values 0...1
- `show_side_by_side(...)` - draws the original image and the filtered image next to each other
- `show_shapes(model, x)` - runs `x` through the model layer by layer and prints every shape

In [ ]:
def get_image(nr=0):
    img, label = digits[nr]
    x = torch.tensor(list(img.getdata()), dtype=torch.float32).reshape(1, 1, 28, 28) / 255
    return x, digits.classes[label]


def show_side_by_side(x, y, title_right):
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(x[0, 0], cmap="gray"); axes[0].set_title("original")
    axes[1].imshow(y[0, 0].detach(), cmap="RdBu_r", vmin=-y.abs().max(), vmax=y.abs().max())
    axes[1].set_title(title_right)
    for ax in axes:
        ax.axis("off")
    plt.show()


def show_shapes(model, x):
    print(f"{'input':12s} {tuple(x.shape)}")
    for layer in model:
        x = layer(x)
        print(f"{type(layer).__name__:12s} {tuple(x.shape)}")
    print("parameters:", sum(p.numel() for p in model.parameters()))


print("helpers ready")

## 💻 Code break 1 - A convolution by numbers
The example from the slide: a 5 × 5 input, a 3 × 3 kernel. `F.conv2d` slides the kernel over the input and
computes "multiply and add" at every position. (PyTorch wants the shape (batch, channels, height, width).)

In [ ]:
x = torch.tensor([[1, 1, 1, 0, 0],
                  [0, 1, 1, 1, 0],
                  [0, 0, 1, 1, 1],
                  [0, 0, 1, 1, 0],
                  [0, 1, 1, 0, 0]], dtype=torch.float32)
kernel = torch.tensor([[1, 0, 1],
                       [0, 1, 0],
                       [1, 0, 1]], dtype=torch.float32)       # 👈 change one number

y = F.conv2d(x.reshape(1, 1, 5, 5), kernel.reshape(1, 1, 3, 3))
print(y[0, 0])

🔧 **Change it:**
1. Set the **middle** number of the kernel to 0. Which outputs change?
2. Set the **top-left** number of the kernel to 0. Which outputs change now?

<details><summary>👀 What you should see</summary>

- original: **[[4, 3, 4], [2, 4, 3], [2, 3, 4]]** - exactly the slide
- middle = 0: every output whose window has a 1 in its centre gets 1 smaller: **[[3, 2, 3], [2, 3, 2], [2, 2, 3]]**
- top-left = 0: **[[3, 2, 3], [2, 3, 2], [2, 3, 3]]** - only outputs whose window had a 1 in its top-left corner change
- the output is 3 × 3: a 3 × 3 kernel fits 3 times into 5 positions (5 - 3 + 1 = 3)
</details>

## 💻 Code break 2 - Hand-made kernels on a real image
Three classic kernels. A CNN *learns* kernels like these by itself in its first layer.

In [ ]:
KERNEL = "vertical"    # 👈 "vertical", "horizontal" or "blur"
IMAGE_NR = 0           # 👈 another image: 1, 2, 3 ...

kernels = {
    "vertical":   torch.tensor([[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]]),   # left minus right
    "horizontal": torch.tensor([[1., 1., 1.], [0., 0., 0.], [-1., -1., -1.]]),   # top minus bottom
    "blur":       torch.ones(3, 3) / 9,                                            # average of 9 pixels
}
x, name = get_image(IMAGE_NR)
y = F.conv2d(x, kernels[KERNEL].reshape(1, 1, 3, 3), padding=1)
print("image:", name, "  input", tuple(x.shape), "  output", tuple(y.shape))
show_side_by_side(x, y, KERNEL + " kernel")

🔧 **Change it:**
1. `KERNEL = "horizontal"`, then `KERNEL = "blur"`. What does each kernel find?
2. `IMAGE_NR = 1` - a different digit.

<details><summary>👀 What you should see</summary>

- image 0 is a handwritten **5**; input and output both **(1, 1, 28, 28)** - `padding=1` keeps the size
- vertical: every stroke gets a **blue left edge** and a **red right edge** - nothing in the flat black background
- horizontal: the **top edges** of the strokes are blue, the **bottom edges** red - strongest in the flat top bar of the 5
- blur: a slightly thicker, blurred copy of the 5 - all red, because there are no negative values
- colours: red = positive output, blue = negative output, white = 0 (nothing found)
- image 1 is a **0**: vertical finds its left and right sides, horizontal its top and bottom
</details>

## 💻 Code break 3 - Shapes: padding, stride, pooling
A convolution layer with 16 filters on a batch of 8 grey images. Watch the shapes.

In [ ]:
KERNEL_SIZE = 3    # 👈
PADDING = 1        # 👈
STRIDE = 1         # 👈

batch = torch.randn(8, 1, 28, 28)                             # (batch, channels, height, width)
conv = nn.Conv2d(1, 16, KERNEL_SIZE, padding=PADDING, stride=STRIDE)
pool = nn.MaxPool2d(2)
print("input:          ", tuple(batch.shape))
print("after Conv2d:   ", tuple(conv(batch).shape))
print("after MaxPool2d:", tuple(pool(conv(batch)).shape))
print("weights of the layer:", tuple(conv.weight.shape))

🔧 **Change it:**
1. `PADDING = 0` - what happens to height and width?
2. `STRIDE = 2` (and `PADDING = 1` again).
3. `KERNEL_SIZE = 5` - which padding keeps the size at 28?

<details><summary>👀 What you should see</summary>

- original: (8, 1, 28, 28) → Conv2d **(8, 16, 28, 28)** → MaxPool2d **(8, 16, 14, 14)**; weights **(16, 1, 3, 3)**
- padding 0: **(8, 16, 26, 26)** - the kernel fits 28 - 3 + 1 = 26 times; after pooling (8, 16, 13, 13)
- stride 2: **(8, 16, 14, 14)** - the kernel jumps 2 pixels → half the size; after pooling (8, 16, 7, 7)
- kernel 5 with padding 1: **(8, 16, 26, 26)**; padding **2** keeps 28 × 28 (rule: padding = kernel_size // 2)
- the channels go from 1 to **16**: one output channel per filter
</details>

## 💻 Code break 4 - A small CNN, layer by layer
The typical recipe: conv → ReLU → pool, twice, then flatten and one linear layer for the 10 classes.

In [ ]:
CHANNELS = 16      # 👈 filters in the first conv layer

model = nn.Sequential(
    nn.Conv2d(1, CHANNELS, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 28 x 28 -> 14 x 14
    nn.Conv2d(CHANNELS, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 14 x 14 -> 7 x 7
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10),                                            # one score per class
)
show_shapes(model, torch.randn(8, 1, 28, 28))
print("for comparison - the day-3 MLP (784 -> 256 -> 128 -> 10): 235146 parameters")

🔧 **Change it:**
1. `CHANNELS = 64` - how many parameters now?
2. Delete the second `nn.MaxPool2d(2)`. Run it - what error do you get, and why?

<details><summary>👀 What you should see</summary>

- shapes: (8, 16, 28, 28) → ReLU → **(8, 16, 14, 14)** → (8, 32, 14, 14) → **(8, 32, 7, 7)** → Flatten **(8, 1568)**
  → **(8, 10)**
- **20 490** parameters - more than 10 × fewer than the MLP from day 3, although it sees the whole image
- 64 channels: **34 794** parameters
- without the second pooling: an **error** "mat1 and mat2 shapes cannot be multiplied (8x6272 and 1568x10)" -
  the image is still 14 × 14, so Flatten gives 32 · 14 · 14 = 6272 numbers, but the Linear layer expects 1568.
  Printing the shapes layer by layer is the fastest way to find such bugs.
</details>